In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Setările implicite ale Transpiler-ului și opțiunile de configurare


<details>
<summary><b>Versiunile pachetelor</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Circuitele abstracte trebuie transpilate deoarece QPU-urile au un set limitat de gate-uri de bază și nu pot executa operații arbitrare. Funcția Transpiler-ului este de a modifica circuitele arbitrare astfel încât să poată rula pe un QPU specificat. Acest lucru se realizează prin traducerea circuitelor în gate-urile de bază suportate și prin introducerea gate-urilor SWAP după cum este necesar, astfel încât conectivitatea Circuit-ului să corespundă celei a QPU-ului.

După cum este explicat în [Transpilează cu manageri de pass](transpile-with-pass-managers), poți crea un [manager de pass](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) folosind funcția [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) și să transmiți un Circuit sau o listă de circuite metodei sale [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) pentru a le transpila. Poți apela `generate_preset_pass_manager` transmițând doar nivelul de optimizare și Backend-ul, alegând să folosești valorile implicite pentru toate celelalte opțiuni, sau poți transmite argumente suplimentare funcției pentru a ajusta fin transpilarea.

## Utilizare de bază fără parametri
În acest exemplu, transmitem un Circuit și un QPU țintă Transpiler-ului fără a specifica niciun alt parametru.

Creează un Circuit și vizualizează rezultatul:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Rezultatul celulei de cod anterioare](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

Transpilează Circuit-ul și vizualizează rezultatul:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Rezultatul celulei de cod anterioare](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## Toți parametrii disponibili
Mai jos sunt toți parametrii disponibili pentru funcția [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). Există două clase de argumente: cele care descriu ținta compilării și cele care influențează modul în care funcționează Transpiler-ul.

Toți parametrii cu excepția `optimization_level` sunt opționali. Pentru detalii complete, consultă [documentația API a Transpiler-ului](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) - Cât de multă optimizare să efectuezi pe circuite. Număr întreg în intervalul (0 - 3). Nivelurile mai ridicate generează circuite mai optimizate, cu costul unui timp de transpilare mai lung. Consultă [Setează nivelul de optimizare al Transpiler-ului](set-optimization) pentru mai multe detalii.

### Parametrii folosiți pentru a descrie ținta compilării:
Aceste argumente descriu QPU-ul țintă pentru execuția Circuit-ului, inclusiv informații precum harta de cuplare a QPU-ului (care descrie conectivitatea qubiților), gate-urile de bază suportate de QPU și ratele de eroare ale gate-urilor.

Mulți dintre acești parametri sunt descriși în detaliu în [Parametrii frecvent utilizați pentru transpilare](common-parameters).

<details>
  <summary>
    **Parametrii QPU (`Backend`)**
  </summary>

**Parametrii Backend** - Dacă specifici `backend`, nu trebuie să specifici `target` sau orice alte opțiuni de backend. De asemenea, dacă specifici `target`, nu trebuie să specifici `backend` sau orice alte opțiuni de backend.
- `backend` (Backend) - Dacă acesta este setat, Transpiler-ul compilează Circuit-ul de intrare pe acest dispozitiv. Dacă orice altă opțiune este setată și afectează aceste setări, cum ar fi `coupling_map`, aceasta suprascrie setările din `backend`.
- `target` (Target) - O țintă Transpiler de Backend. De obicei aceasta este specificată ca parte a argumentului backend, dar dacă ai construit manual un obiect Target, îl poți specifica aici. Aceasta suprascrie ținta din `backend`.
- `backend_properties` (BackendProperties) - Proprietăți returnate de un QPU, inclusiv informații despre erorile gate-urilor, erorile de citire, timpii de coerență ai qubiților și altele. Găsește un QPU care furnizează aceste informații rulând `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) - O restricție opțională a hardware-ului de control privind rezoluția timpului de instrucțiune. Aceste informații sunt furnizate de configurația QPU-ului. Dacă QPU-ul nu are nicio restricție privind alocarea timpului de instrucțiune, `timing_constraints` este `None` și nu se efectuează nicio ajustare. Un QPU poate raporta un set de restricții, și anume:
    - `granularity`: O valoare întreagă reprezentând rezoluția minimă a gate-ului de puls în unități de dt. Un gate de puls definit de utilizator ar trebui să aibă o durată care este un multiplu al acestei valori de granularitate.
    - `min_length`: O valoare întreagă reprezentând lungimea minimă a gate-ului de puls în unități de dt. Un gate de puls definit de utilizator ar trebui să fie mai lung decât această lungime.
    - `pulse_alignment`: O valoare întreagă reprezentând rezoluția timpului de pornire a instrucțiunii gate. Instrucțiunile gate ar trebui să înceapă la un moment care este un multiplu al acestei valori.
    - `acquire_alignment`: O valoare întreagă reprezentând rezoluția timpului de pornire a instrucțiunii de măsurare. Instrucțiunea de măsurare ar trebui să înceapă la un moment care este un multiplu al acestei valori.
</details>

<details>
  <summary>
    **Parametrii de layout și topologie**
  </summary>

- `basis_gates` (List[str] | None) - Lista numelor gate-urilor de bază la care să se deruleze. De exemplu ['u1', 'u2', 'u3', 'cx']. Dacă este `None`, nu se derulează.
- `coupling_map` (CouplingMap | List[List[int]]) - Hartă de cuplare direcționată (posibil personalizată) pentru a o viza în mapare. Dacă harta de cuplare este simetrică, ambele direcții trebuie specificate. Aceste formate sunt suportate:
    - Instanță CouplingMap
    - Listă - trebuie furnizată ca matrice de adiacență, unde fiecare intrare specifică toate interacțiunile direcționate cu doi qubiți suportate de QPU. De exemplu: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - Maparea operațiilor de Circuit la programele de puls. Dacă este `None`, se folosește `instruction_schedule_map` al QPU-ului.
</details>

### Parametrii folosiți pentru a influența modul în care funcționează Transpiler-ul
Acești parametri afectează etapele specifice de transpilare. Unii dintre ei pot afecta mai multe etape, dar au fost listați doar sub o singură etapă pentru simplitate. Dacă specifici un argument, cum ar fi `initial_layout` pentru qubiții pe care vrei să îi folosești, acea valoare suprascrie toate pass-urile care ar putea să o modifice. Cu alte cuvinte, Transpiler-ul nu va schimba nimic ce specifici manual. Pentru detalii despre etapele specifice, consultă [Etapele Transpiler-ului](transpiler-stages).

<details>
  <summary>
    **Etapa de inițializare**
  </summary>

- `hls_config` (HLSConfig) - O clasă de configurare opțională `HLSConfig` care este transmisă direct pass-ului de transformare `HighLevelSynthesis`. Această clasă de configurare îți permite să specifici listele de algoritmi de sinteză și parametrii acestora pentru diverse obiecte de nivel înalt.
- `init_method` (str) - Numele plugin-ului de utilizat pentru etapa de inițializare. În mod implicit, nu se folosește un plugin extern. Poți vedea o listă a plugin-urilor instalate rulând `list_stage_plugins()` cu `init` ca argument pentru numele etapei.
- `unitary_synthesis_method` (str) - Numele metodei de sinteză unitară de utilizat. În mod implicit, se folosește `default`. Poți vedea o listă a plugin-urilor instalate rulând `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) - Un dicționar de configurare opțional care este transmis direct plugin-ului de sinteză unitară. În mod implicit, această setare nu are niciun efect deoarece metoda implicită de sinteză unitară nu acceptă configurare personalizată. Aplicarea unei configurații personalizate ar trebui să fie necesară doar când un plugin de sinteză unitară este specificat cu argumentul `unitary_synthesis`. Deoarece acesta este personalizat pentru fiecare plugin de sinteză unitară, consultă documentația plugin-ului pentru modul de utilizare a acestei opțiuni.
</details>

<details>
  <summary>
    **Etapa de layout**
  </summary>

- `initial_layout` (Layout | Dict | List) - Poziția inițială a qubiților virtuali pe qubiții fizici. Dacă acest layout face Circuit-ul compatibil cu constrângerile `coupling_map`, acesta va fi utilizat. Layout-ul final nu este garantat să fie același, deoarece Transpiler-ul ar putea permuta qubiții prin swap-uri sau alte mijloace. Pentru detalii complete, consultă [secțiunea Layout inițial.](common-parameters#initial-layout)
- `layout_method` (str) - Numele pass-ului de selecție a layout-ului (`default`, `dense`, `sabre` și `trivial`). Acesta poate fi și numele plugin-ului extern de utilizat pentru etapa de layout. Poți vedea o listă a plugin-urilor instalate rulând `list_stage_plugins()` cu `layout` pentru argumentul `stage_name`. Valoarea implicită este `sabre`.
</details>

<details>
  <summary>
    **Etapa de rutare**
  </summary>

- `routing_method` (str) - Numele pass-ului de rutare (`basic`, `lookahead`, `default`, `sabre` sau `none`). Acesta poate fi și numele plugin-ului extern de utilizat pentru etapa de rutare. Poți vedea o listă a plugin-urilor instalate rulând `list_stage_plugins()` cu `routing` pentru argumentul `stage_name`. Valoarea implicită este `sabre`.
</details>

<details>
  <summary>
    **Etapa de traducere**
  </summary>

- `translation_method` (str) - Numele pass-ului de traducere (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). Acesta poate fi și numele plugin-ului extern de utilizat pentru etapa de traducere. Poți vedea o listă a plugin-urilor instalate rulând `list_stage_plugins()` cu `translation` pentru argumentul `stage_name`. Valoarea implicită este `translator`.
</details>

<details>
  <summary>
    **Etapa de optimizare**
  </summary>

- `approximation_degree` (float, în intervalul 0-1 | None) - Selector euristic folosit pentru aproximarea Circuit-ului (1.0 = nicio aproximare, 0.0 = aproximare maximă). Valoarea implicită este 1.0. Specificând `None` se setează gradul de aproximare la rata de eroare raportată. Consultă [secțiunea Grad de aproximare](common-parameters#approx-degree) pentru mai multe detalii.
- `optimization_method` (str) - Numele plugin-ului de utilizat pentru etapa de optimizare. În mod implicit, nu se folosește un plugin extern. Poți vedea o listă a plugin-urilor instalate rulând `list_stage_plugins()` cu `optimization` pentru argumentul `stage_name`.
</details>

<details>
  <summary>
    **Etapa de planificare**
  </summary>

- `scheduling_method` (str) - Numele pass-ului de planificare. Acesta poate fi și numele plugin-ului extern de utilizat pentru etapa de planificare. Poți vedea o listă a plugin-urilor instalate rulând `list_stage_plugins()` cu `scheduling` pentru argumentul `stage_name`.
  * 'as_soon_as_possible': Planifică instrucțiunile lacom, cât mai devreme posibil pe o resursă qubit (alias: `asap`).
  * 'as_late_as_possible': Planifică instrucțiunile târziu, adică menținând qubiții în starea de bază când este posibil (alias: `alap`). Aceasta este valoarea implicită.
</details>

<details>
  <summary>
    **Execuția Transpiler-ului**
  </summary>

- `seed_transpiler` (int) - Setează semințele aleatoare pentru părțile stocastice ale Transpiler-ului.
</details>

Următoarele valori implicite sunt utilizate dacă nu specifici niciun parametru de mai sus. Consultă [pagina de referință API](../api/qiskit/transpiler_preset) a metodei pentru mai multe informații:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>